# Contract protocol v2 sketchboard

*Agents are **not** allowed to edit this notebook unless explicitly asked for.*

Fix a horizon and let the admissible start set be

$$
I=\{0.1,0.2,\ldots,\text{horizon}\}.
$$

Let

$$
C^{\mathrm{cmt}}=\langle \top \mid K\rangle,
\qquad
C^{\mathrm{req}}=\left\langle \top \mid \exists_I H^d p\right\rangle,
\quad p\equiv(j=k).
$$

Define the **probe strengthening**

$$
\hat C^{\mathrm{req}}
=
\left\langle
\top
\mid
\forall_I H^d p
\right\rangle
\equiv
\left\langle
\top
\mid
\forall_{I\oplus[0,d]}p
\right\rangle.
$$

Then:

$$
C^{\mathrm{cand}}
=
C^{\mathrm{cmt}}\bullet \hat C^{\mathrm{req}}.
$$

If $C^{\mathrm{cand}}$ is consistent, generate concrete refinements of the original request:

$$
\mathcal O
=
\left\{
\left\langle\top\mid\forall_{J'}p\right\rangle
:
J'=[s,s+d),\ s\in I
\right\}.
$$

If $C^{\mathrm{cand}}$ is inconsistent, apply the old repair path to the strengthened envelope:

$$
\forall_{I\oplus[0,d]}p
\;\xrightarrow{\text{weaken}}\;
\exists_{I\oplus[0,d]}p
$$

$$
C^s
=
C^{w}\div C^{\mathrm{cmt}},
\qquad
C^{ms}
=
C^s\bullet C^{\mathrm{cmt}}.
$$

After projection/postprocessing, obtain the surviving availability

$$
\boxed{
\exists_J p,
\qquad
J\subseteq I\oplus[0,d].
}
$$

Then **concretize** it into commitment-ready offers:

$$
\boxed{
\mathcal O
=
\left\{
\left\langle\top\mid\forall_{J'_r}p\right\rangle
:
J'_r=[s,s+d),\quad s\in I,\quad J'_r\subseteq J
\right\}.
}
$$

Thus every offer starts on the admissible $0.1$ grid and preserves the requested duration $d$.

So, compactly:

$$
\boxed{
\exists_I H^d p
\;\xrightarrow{\text{probe}}\;
\forall_I H^d p
\;\xrightarrow{\text{merge}}
\begin{cases}
\text{success} &\to \text{concrete duration-}d\text{ offers},\\
\text{failure} &\to
\text{weaken}\to\text{separate}\to\text{merge}
\to \exists_Jp
\to\text{concrete offers}.
\end{cases}
}
$$

In [1]:

from dataclasses import dataclass
from fractions import Fraction
from steps import *

import portion as P
import interval_logic as il


@dataclass
class Contract:
    assumption: il.Formula | bool | str
    guarantee: il.Formula | bool | str

    def __post_init__(self) -> None:
        self.assumption = il.to_nnf(self.assumption).simplify()

        # Maintain saturated-form invariant.
        self.guarantee = il.to_nnf(il.Or((
            il.as_formula(self.guarantee),
            il.Not(self.assumption),
        ))).simplify()

    @staticmethod
    def _s(
        assumption: il.Formula | bool | str,
        guarantee: il.Formula | bool | str,
    ) -> str:
        return f"⟨ {il.as_formula(assumption)} ∣ {il.as_formula(guarantee)} ⟩"

    def __str__(self) -> str:
        return self._s(self.assumption, self.guarantee)

    @staticmethod
    def merge_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⊗ {other!s}"

    def merge(self, other: "Contract") -> "Contract":
        return Contract(
            il.And((self.assumption, other.assumption)),
            il.Or((
                il.And((self.guarantee, other.guarantee)),
                il.Not(il.And((self.assumption, other.assumption))),
            )),
        )

    @staticmethod
    def separate_s(self: "Contract", other: "Contract") -> str:
        return f"{self!s} ⦼ {other!s}"

    def separate(self, other: "Contract") -> "Contract":
        return Contract(
            il.Or((
                il.And((self.assumption, other.guarantee)),
                il.Not(il.And((self.guarantee, other.assumption))),
            )),
            il.And((self.guarantee, other.assumption)),
        )

    def to_nnf(self) -> "Contract":
        return Contract(
            il.to_nnf(self.assumption),
            il.to_nnf(self.guarantee),
        )

    def simplify(self) -> "Contract":
        return Contract(
            self.assumption.simplify(),
            self.guarantee.simplify(),
        )

    def replace(self, *fns) -> "Contract":
        return Contract(
            self.assumption.replace(*fns),
            self.guarantee.replace(*fns),
        )

## Request and offer objects

`Request.I` is the finite set of allowed starts $\{0.1,0.2,\ldots,\text{horizon}\}$.

`Request.envelope` is $I\oplus[0,d]$, and `Offer.interval` is a concrete interval $[s,s+d)$ with $s\in I$.


In [2]:

GRID_STEP = Fraction(1, 10)


def exact_time(value: il.TimeValue) -> Fraction:
    return il.RelativeParameter(value).duration


def start_grid(
    horizon: il.TimeValue,
    *,
    step_size: il.TimeValue = GRID_STEP,
) -> tuple[Fraction, ...]:
    horizon = exact_time(horizon)
    step_size = exact_time(step_size)
    if step_size == 0:
        raise ValueError("step_size must be positive")

    return tuple(
        n * step_size
        for n in range(1, int(horizon // step_size) + 1)
    )


@dataclass(frozen=True)
class Request:
    job: int
    duration: il.TimeValue
    horizon: il.TimeValue

    def __post_init__(self) -> None:
        duration = exact_time(self.duration)
        if duration == 0:
            raise ValueError("duration must be positive")
        object.__setattr__(self, "duration", duration)
        object.__setattr__(self, "horizon", exact_time(self.horizon))

    @property
    def I(self) -> tuple[Fraction, ...]:
        return start_grid(self.horizon)

    def window(self, start: il.TimeValue) -> P.Interval:
        start = exact_time(start)
        return P.closedopen(start, start + self.duration)

    @property
    def envelope(self) -> P.Interval:
        result = P.empty()
        for start in self.I:
            result |= self.window(start)
        return result

    @property
    def proposition(self) -> il.Formula:
        return il.Signal("j").equals(self.job)

    @property
    def contract(self) -> Contract:
        # The AST requires connected absolute parameters, so the finite
        # existential over I is represented as one singleton witness per s.
        return Contract(True, il.Or(
            il.Eventually(
                il.Always(self.proposition, self.duration),
                P.singleton(start),
            )
            for start in self.I
        ))


@dataclass(frozen=True)
class Offer:
    job: int
    duration: il.TimeValue
    interval: P.Interval

    @property
    def contract(self) -> Contract:
        return Contract(
            True,
            il.Always(
                il.Signal("j").equals(self.job),
                self.interval,
            ),
        )


## Protocol transformations

Relative to target job `k`, the projection forgets which *other* job occupies the charger:

- `j = k` becomes `p`,
- `j = n`, `n != k`, becomes `¬p`.

This is a scheduler projection, not a generic logical equivalence.


In [3]:

p = il.Proposition("p")
j = il.Signal("j")


def project_job(job: int):
    target = il.NaturalLiteral(job)

    def replacer(psi):
        match psi:
            case il.Equal(left, right):
                if left == j:
                    return p if right == target else il.Not(p)
                if right == j:
                    return p if left == target else il.Not(p)

        return psi

    return replacer


def eventually_over(phi: il.Formula, interval: P.Interval) -> il.Formula:
    # Absolute temporal parameters are connected, so a disconnected set
    # is represented by a disjunction over its connected components.
    return il.Or(il.Eventually(phi, component) for component in interval)


def always_over(phi: il.Formula, interval: P.Interval) -> il.Formula:
    return il.And(il.Always(phi, component) for component in interval)


def probe_strengthen(request: Request, phi: il.Formula) -> Contract:
    # ∀_I H^d phi ≡ ∀_(I ⊕ [0,d]) phi.
    return Contract(True, always_over(phi, request.envelope))


def weaken_strengthened(request: Request, phi: il.Formula) -> Contract:
    # ∀_(I ⊕ [0,d]) phi -> ∃_(I ⊕ [0,d]) phi.
    return Contract(True, eventually_over(phi, request.envelope))


def overlap(psi):
    # ∀_I ¬p ∧ ∃_J p -> ∀_I ¬p ∧ ∃_(J-I) p
    match psi:
        case il.And((
            il.Always(phi, il.AbsoluteParameter(I)),
            il.Eventually(q, il.AbsoluteParameter(J)),
        )):
            base = q.body if isinstance(q, il.Always) else q
            if phi == il.Not(base):
                return il.And((
                    il.Always(phi, I),
                    eventually_over(q, J - I),
                ))

        case il.And((
            il.Eventually(q, il.AbsoluteParameter(J)),
            il.Always(phi, il.AbsoluteParameter(I)),
        )):
            base = q.body if isinstance(q, il.Always) else q
            if phi == il.Not(base):
                return il.And((
                    il.Always(phi, I),
                    eventually_over(q, J - I),
                ))

    return psi


def postprocess_repair(
    C_ms: Contract,
    job: int,
    envelope: P.Interval,
    unavailable: P.Interval,
) -> Contract:
    C_post = Contract(
        True,
        C_ms.guarantee & C_ms.assumption,
    )

    C_post = C_post.replace(project_job(job)).simplify()
    C_post = C_post.replace(overlap).simplify()

    # Collapse the scheduler-specific projection to its canonical
    # surviving availability, including disconnected components.
    surviving = envelope - unavailable
    return Contract(True, eventually_over(p, surviving))


## Offer generation

Offers are generated only at starts $s\in I$. Each offer is exactly $J'_r=[s,s+d)$; on the repair branch it must also satisfy $J'_r\subseteq J$.


In [4]:

def generate_offers(
    request: Request,
    available: P.Interval | None = None,
) -> list[Offer]:
    offers = []
    for start in request.I:
        J = request.window(start)
        if available is None or (J - available).empty:
            offers.append(Offer(request.job, request.duration, J))
    return offers


def find_available_interval(
    expression: il.Formula,
) -> P.Interval | None:
    found = None

    def visit(psi):
        nonlocal found

        match psi:
            case il.Eventually(phi, il.AbsoluteParameter(I)):
                if phi == p:
                    found = I if found is None else found | I

        return psi

    expression.replace(visit)
    return found


# Case A — strengthened probe fails

Existing commitment:

$$
C^{cmt}
=
\langle \top \mid \forall_{[1,3)}(j=1) \rangle.
$$

Flexible request with $d=1$, horizon $=4$, and $I=\{0.1,0.2,\ldots,4.0\}$:

$$
C^{req}_2
=
\langle \top \mid \exists_I H^1(j=2) \rangle.
$$

Its strengthened envelope intersects the existing commitment, so the failed-probe repair branch applies.


In [5]:

reset_step()

with step("Protocol — failed strengthened probe"):

    println("Commitments")
    reserved = P.closedopen(1, 3)
    printeq(
        C_cmt := Contract(True, il.Always(j.equals(1), reserved)),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        horizon=4.0,
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := probe_strengthen(req, p),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        probe_consistent = (reserved & req.envelope).empty
        assert not probe_consistent
        println(
            f"Probe consistent: {probe_consistent}",
            "→ continue to repair availability",
            top=1,
        )

    with step("Repair availability"):

        println("Weaken")
        printeq(
            C_str,
            C_w := weaken_strengthened(req, p),
        )

        println("Separate", top=1)
        printeq(
            Contract.separate_s(C_w, C_cmt_p),
            C_s := C_w.separate(C_cmt_p),
        )

        println("Merge separated", top=1)
        printeq(
            Contract.merge_s(C_s, C_cmt_p),
            C_ms := C_s.merge(C_cmt_p),
        )

        println("Postprocess / project", top=1)
        printeq(
            Contract._s(True, C_ms.guarantee & C_ms.assumption),
            C_post := postprocess_repair(
                C_ms, req.job, req.envelope, reserved
            ),
        )

    with step("Concretize"):
        available = find_available_interval(C_post.guarantee)

        println(f"Available region: {available}")

        offers = generate_offers(req, available)

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


⏺ Step 1: Protocol — failed strengthened probe
│ Commitments
│ ⟨ ⊤ ∣ □_[1,3) j = 1 ⟩
│ = ⟨ ⊤ ∣ □_[1,3) ¬p ⟩ 
│ 
│ Request
│ ⟨ ⊤ ∣ ◇_[0.1,0.1] □_1 j = 2 ∨ ◇_[0.2,0.2] □_1 j = 2 ∨ ◇_[0.3,0.3] □_1 j = 2 ∨ ◇_[0.4,0.4] □_1 j = 2 ∨ ◇_[0.5,0.5] □_1 j = 2 ∨ ◇_[0.6,0.6] □_1 j = 2 ∨ ◇_[0.7,0.7] □_1 j = 2 ∨ ◇_[0.8,0.8] □_1 j = 2 ∨ ◇_[0.9,0.9] □_1 j = 2 ∨ ◇_[1,1] □_1 j = 2 ∨ ◇_[1.1,1.1] □_1 j = 2 ∨ ◇_[1.2,1.2] □_1 j = 2 ∨ ◇_[1.3,1.3] □_1 j = 2 ∨ ◇_[1.4,1.4] □_1 j = 2 ∨ ◇_[1.5,1.5] □_1 j = 2 ∨ ◇_[1.6,1.6] □_1 j = 2 ∨ ◇_[1.7,1.7] □_1 j = 2 ∨ ◇_[1.8,1.8] □_1 j = 2 ∨ ◇_[1.9,1.9] □_1 j = 2 ∨ ◇_[2,2] □_1 j = 2 ∨ ◇_[2.1,2.1] □_1 j = 2 ∨ ◇_[2.2,2.2] □_1 j = 2 ∨ ◇_[2.3,2.3] □_1 j = 2 ∨ ◇_[2.4,2.4] □_1 j = 2 ∨ ◇_[2.5,2.5] □_1 j = 2 ∨ ◇_[2.6,2.6] □_1 j = 2 ∨ ◇_[2.7,2.7] □_1 j = 2 ∨ ◇_[2.8,2.8] □_1 j = 2 ∨ ◇_[2.9,2.9] □_1 j = 2 ∨ ◇_[3,3] □_1 j = 2 ∨ ◇_[3.1,3.1] □_1 j = 2 ∨ ◇_[3.2,3.2] □_1 j = 2 ∨ ◇_[3.3,3.3] □_1 j = 2 ∨ ◇_[3.4,3.4] □_1 j = 2 ∨ ◇_[3.5,3.5] □_1 j = 2 ∨ ◇_[3.6,3.6] □_1 j = 2 ∨ ◇_[3.7,3.7] □_1 j 

# Case B — strengthened probe succeeds

Here the existing commitment $[6,8)$ starts after the request envelope. The request has $d=1$, horizon $=5$, and $I=\{0.1,0.2,\ldots,5.0\}$:

$$
C^{req}_2
=
\langle \top \mid \exists_I H^1(j=2) \rangle.
$$

Because the strengthened probe is consistent, the repair path is skipped and concrete offers are generated directly from the original flexible request.


In [6]:

reset_step()

with step("Protocol — successful strengthened probe"):

    println("Commitments")
    reserved = P.closedopen(6, 8)
    printeq(
        C_cmt := Contract(True, il.Always(j.equals(1), reserved)),
        C_cmt_p := C_cmt.replace(project_job(2)),
    )

    println("Request", top=1)
    req = Request(
        job=2,
        duration=1.0,
        horizon=5.0,
    )
    printeq(
        C_req := req.contract,
        C_req_p := C_req.replace(project_job(req.job)),
    )

    with step("Probe strengthen"):
        printeq(
            C_req_p,
            C_str := probe_strengthen(req, p),
        )

    with step("Merge strengthened"):
        printeq(
            Contract.merge_s(C_cmt_p, C_str),
            C_probe := C_cmt_p.merge(C_str),
        )

        probe_consistent = (reserved & req.envelope).empty
        assert probe_consistent
        println(
            f"Probe consistent: {probe_consistent}",
            "→ skip repair",
            top=1,
        )

    with step("Concretize original request"):
        offers = generate_offers(req)

        for n, offer in enumerate(offers, 1):
            println(
                f"Offer {n}",
                str(offer.contract),
            )


⏺ Step 1: Protocol — successful strengthened probe
│ Commitments
│ ⟨ ⊤ ∣ □_[6,8) j = 1 ⟩
│ = ⟨ ⊤ ∣ □_[6,8) ¬p ⟩ 
│ 
│ Request
│ ⟨ ⊤ ∣ ◇_[0.1,0.1] □_1 j = 2 ∨ ◇_[0.2,0.2] □_1 j = 2 ∨ ◇_[0.3,0.3] □_1 j = 2 ∨ ◇_[0.4,0.4] □_1 j = 2 ∨ ◇_[0.5,0.5] □_1 j = 2 ∨ ◇_[0.6,0.6] □_1 j = 2 ∨ ◇_[0.7,0.7] □_1 j = 2 ∨ ◇_[0.8,0.8] □_1 j = 2 ∨ ◇_[0.9,0.9] □_1 j = 2 ∨ ◇_[1,1] □_1 j = 2 ∨ ◇_[1.1,1.1] □_1 j = 2 ∨ ◇_[1.2,1.2] □_1 j = 2 ∨ ◇_[1.3,1.3] □_1 j = 2 ∨ ◇_[1.4,1.4] □_1 j = 2 ∨ ◇_[1.5,1.5] □_1 j = 2 ∨ ◇_[1.6,1.6] □_1 j = 2 ∨ ◇_[1.7,1.7] □_1 j = 2 ∨ ◇_[1.8,1.8] □_1 j = 2 ∨ ◇_[1.9,1.9] □_1 j = 2 ∨ ◇_[2,2] □_1 j = 2 ∨ ◇_[2.1,2.1] □_1 j = 2 ∨ ◇_[2.2,2.2] □_1 j = 2 ∨ ◇_[2.3,2.3] □_1 j = 2 ∨ ◇_[2.4,2.4] □_1 j = 2 ∨ ◇_[2.5,2.5] □_1 j = 2 ∨ ◇_[2.6,2.6] □_1 j = 2 ∨ ◇_[2.7,2.7] □_1 j = 2 ∨ ◇_[2.8,2.8] □_1 j = 2 ∨ ◇_[2.9,2.9] □_1 j = 2 ∨ ◇_[3,3] □_1 j = 2 ∨ ◇_[3.1,3.1] □_1 j = 2 ∨ ◇_[3.2,3.2] □_1 j = 2 ∨ ◇_[3.3,3.3] □_1 j = 2 ∨ ◇_[3.4,3.4] □_1 j = 2 ∨ ◇_[3.5,3.5] □_1 j = 2 ∨ ◇_[3.6,3.6] □_1 j = 2 ∨ ◇_[3.7,3.7] □_

## Next iteration

One piece intentionally remains outside the contract algebra:

- **General consistency:** the examples can decide the branch by intersecting their explicit occupied intervals. A general implementation still needs a scheduler satisfiability check over natural-number signals.

The contract operations expose the temporal resource that survives; `generate_offers` then returns exactly the admissible fixed-duration `Always` contracts.
